In [ ]:
import polars as pl

In [2]:
# 1) Load the country dictionary into a LazyFrame using scan in order to prevent RAM saturation:
countries = pl.scan_csv("../data/country_codes_V202601.csv")

In [3]:
# 2) Load the specific year into a LazyFrame using scan in order to prevent RAM saturation:
df = pl.scan_csv("../data/raw/BACI_HS92_Y2024_V202601.csv")

In [4]:
# 3.1) Join the country code of the importers in df with the countries df to get the country ISO3 code: 
df = df.join(countries, left_on='i', right_on='country_code').rename({'country_iso3': 'Importer'})

# 3.2) Join the country code of the exporters in df with the countries df to get the country ISO3 code: 
df = df.join(countries, left_on='j', right_on='country_code').rename({'country_iso3': 'Exporter'})

# 3.3) Select the necessary columns and rename them: 
df = df.select([
        pl.col("t").alias("Year"),
        pl.col("Exporter"),
        pl.col("Importer"),
        pl.col("k").alias("Product_Code"),
        pl.col("v").alias("Value_Thousands_USD")
    ]).collect()

In [5]:
# 4) Data type adjustment: 
df = df.with_columns([pl.col('Product_Code').cast(pl.UInt32), 
                      pl.col('Value_Thousands_USD').cast(pl.Float32), 
                      pl.col('Year').cast(pl.UInt16), 
                      pl.col('Exporter').cast(pl.Categorical), 
                      pl.col('Importer').cast(pl.Categorical)])

In [ ]:
# 5) View the new DataFrame: 
df

Year,Exporter,Importer,Product_Code,Value_Thousands_USD
u16,cat,cat,u32,f32
2024,"""AGO""","""AFG""",80810,0.176
2024,"""AGO""","""AFG""",330499,2.295
2024,"""AGO""","""AFG""",732510,5.617
2024,"""AGO""","""AFG""",848330,2.42
2024,"""AGO""","""AFG""",853610,0.605
…,…,…,…,…
2024,"""URY""","""ZMB""",610429,0.585
2024,"""URY""","""ZMB""",848490,0.048
2024,"""URY""","""ZMB""",870810,0.02


In [ ]:
# 5) Implement it as a function: 

def process_year(year: str | int, version: str = "V202601") -> pl.DataFrame:
    """Load, clean, and normalize BACI trade data for a single year."""
    
    year = int(year)

    countries = (
        pl.scan_csv(f"../data/country_codes_{version}.csv")
        .select(["country_code", "country_iso3"])
    )

    df = (
        pl.scan_csv(f"../data/raw/BACI_HS92_Y{year}_{version}.csv")
        .join(countries, left_on="i", right_on="country_code")
        .rename({"country_iso3": "Importer"})
        .join(countries, left_on="j", right_on="country_code")
        .rename({"country_iso3": "Exporter"})
        .select(
            [
                pl.col("t").alias("Year"),
                pl.col("Exporter"),
                pl.col("Importer"),
                pl.col("k").alias("Product_Code"),
                pl.col("v").alias("Value_Thousands_USD"),
            ]
        )
        .with_columns(
            [
                pl.col("Product_Code").cast(pl.UInt32),
                pl.col("Value_Thousands_USD").cast(pl.Float32),
                pl.col("Year").cast(pl.UInt16),
                pl.col("Exporter").cast(pl.Categorical),
                pl.col("Importer").cast(pl.Categorical),
            ]
        )
        .collect()
    )

    return df

In [ ]:
from pathlib import Path
import re

def process_all(
    start_year: int | None = None,
    end_year: int | None = None,
    version: str = "V202601",
) -> pl.DataFrame:
    """Process all available BACI yearly files and concatenate them."""

    raw_dir = Path("../data/raw")
    year_pattern = re.compile(r"_Y(\d{4})_")

    years = []
    for file_path in raw_dir.glob(f"BACI_HS92_Y*_{version}.csv"):
        match = year_pattern.search(file_path.name)
        if match:
            year = int(match.group(1))
            if (start_year is None or year >= start_year) and (end_year is None or year <= end_year):
                years.append(year)

    years = sorted(set(years))

    if not years:
        raise ValueError(
            f"No input files found for version={version}, start_year={start_year}, end_year={end_year}."
        )

    frames = [process_year(year, version=version) for year in years]
    return pl.concat(frames, how="vertical_relaxed")